In [ ]:
import torch
import torch.nn.functional as f
from torchvision import transforms
from PIL import Image, ImageDraw

from src.models.Yolo import Yolo

from src.cnn.VisDroneCNN import VisDroneCNN
from src.dataset.Dataset import VisDrone
from src.dataset.collate import collate_fn

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((1024, 1024))
])

### Build model
Load model and make prediction for given images

In [ ]:
train_dev_data = "data/VisDrone_Dataset/VisDrone2019-DET-test-dev/images"
train_labels = "data/VisDrone_Dataset/VisDrone2019-DET-test-dev/labels"

visdrone_test = VisDrone(data_dir=train_dev_data, labels_dir=train_labels, transform=transform)
dataset = torch.utils.data.DataLoader(visdrone_test,
                                      batch_size=8,
                                      shuffle=True,
                                      num_workers=4,
                                      collate_fn=collate_fn,
                                      pin_memory=True,
                                      prefetch_factor=2)


cnn = VisDroneCNN(S=16, B_boxes=1, C=10)
cnn.load_state_dict(torch.load('first.pth'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

images, labels = next(iter(dataset))

#cnn.to(device)
#images.to(device)
with torch.no_grad():
    output: torch.Tensor = cnn(images)

In [ ]:
prediction = output[0]

### Additional Transforms
Additional transformation for input data. It's necessary because model works on data without transformation like softmax or sigmoid in outer layer (It's done during training in loss function).

In [ ]:
def decode_prediction(models_prediction: torch.Tensor) -> tuple:
    batch_center_x = torch.sigmoid(models_prediction[..., 0])
    batch_center_y = torch.sigmoid(models_prediction[..., 1])
    batch_box_width = torch.sigmoid(models_prediction[..., 2])
    batch_box_height = torch.sigmoid(models_prediction[..., 3])
    batch_objectness = torch.sigmoid(models_prediction[..., 4])
    batch_classes_prob = f.softmax(models_prediction[..., 5:], dim=-1)
    return batch_center_x, batch_center_y, batch_box_width, batch_box_height, batch_objectness, batch_classes_prob

In [ ]:
cnn.eval()
with torch.no_grad():
    bx, by, bw, bh, obj, cls = decode_prediction(models_prediction=output)
    print("obj min/max:", obj.min().item(), obj.max().item())

    print("cls sum in single cell:", cls[0, 0, 0].sum().item())

    print("bx min/max:", bx.min().item(), bx.max().item())

In [ ]:
def models_output_to_yolo(batch_center_x: torch.Tensor,
                          batch_center_y: torch.Tensor,
                          batch_box_width: torch.Tensor,
                          batch_box_height: torch.Tensor,
                          img_w: int = 256,
                          img_h: int = 256) -> Yolo:
    grid = batch_center_x.shape[-1]
    cell_w = img_w / grid
    cell_h = img_h / grid

    offset_x = torch.arange(grid).float().view(1, 1, grid).expand(bx.shape[0], grid, grid)
    offset_y = torch.arange(grid).float().view(1, grid, 1).expand(by.shape[0], grid, grid)

    cx = (offset_x + batch_center_x) * cell_w
    cy = (offset_y + batch_center_y) * cell_h

    bw_px = batch_box_width * img_w
    bh_px = batch_box_height * img_h

    x1 = cx - bw_px / 2
    x2 = cx + bw_px / 2
    y1 = cy - bh_px / 2
    y2 = cy + bh_px / 2

    boxes = torch.stack([x1, y1, x2, y2], dim=-1)

    output_converted_to_pixel_vals = Yolo(boxes=boxes, labels=torch.zeros_like(boxes))

    return output_converted_to_pixel_vals

In [ ]:
yolo = models_output_to_yolo(bx, by, bw, bh, img_w=256, img_h=256)

print("boxes.shape:", yolo.boxes.shape)          # [B, S, S, 4]
print("x1 min/max:", yolo.boxes[..., 0].min().item(), yolo.boxes[..., 0].max().item())
print("y1 min/max:", yolo.boxes[..., 1].min().item(), yolo.boxes[..., 1].max().item())
print("x2 min/max:", yolo.boxes[..., 2].min().item(), yolo.boxes[..., 2].max().item())
print("y2 min/max:", yolo.boxes[..., 3].min().item(), yolo.boxes[..., 3].max().item())

### Filtering
We have to filter garbage predictions

In [ ]:
def filter_predictions(boxes: torch.Tensor,
                       batch_objectness: torch.Tensor,
                       batch_classes_prob: torch.Tensor,
                       conf_threshold: float = 0.5) ->list[dict]:
    batch_size = boxes.shape[0]
    filtered_boxes = []
    for box in range(batch_size):
        class_scores, class_ids = batch_classes_prob[box].max(dim=-1)
        scores = batch_objectness[box] * class_scores

        mask = scores > conf_threshold

        filtered_boxes.append({
            "boxes": boxes[box][mask],
            "scores": scores[mask],
            "class_ids": class_ids[mask]
        })
    return results

In [ ]:
results = filter_predictions(yolo.boxes, obj, cls, conf_threshold=0.55)

for b, r in enumerate(results):
    print(f"Obraz {b}: {len(r['boxes'])} boxes after filtering")

In [ ]:
CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van",
    "truck", "tricycle", "awning-tricycle", "bus", "motor"
]

COLORS = [
    (255, 56,  56),  (255, 157, 151), (255, 112,  31),
    (255, 178,  29), (207, 210,  49), (72,  249,  10),
    (146, 204,  23), (61,  219, 134), (26,  147,  52),
    (0,  212, 187),
]

def draw_predictions(image: Image.Image,
                     result: dict) -> Image.Image:
    draw = ImageDraw.Draw(image)

    for box, score, cls_id in zip(result["boxes"], result["scores"], result["class_ids"]):
        x1, y1, x2, y2 = box.tolist()
        cls_id = int(cls_id)
        color  = COLORS[cls_id]
        label  = f"{CLASS_NAMES[cls_id]}: {score:.2f}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)

        text_bbox = draw.textbbox((x1, y1), label)
        draw.rectangle(text_bbox, fill=color)
        draw.text((x1, y1), label, fill=(255, 255, 255))

    return image

In [ ]:
img_np = images[0].permute(1, 2, 0).cpu().numpy()
img_np = (img_np * 255).clip(0, 255).astype("uint8")
pil_img = Image.fromarray(img_np)

pil_img = draw_predictions(pil_img, results[0])
pil_img.show()